# 📊 Retail Time Series ML Pipeline

**Complete, runnable walkthrough** of the 10-step ML pipeline.

Each section follows [docs/ml_pipeline_guide.md](../docs/ml_pipeline_guide.md) with reasoning.

> Run cells top-to-bottom. Each step builds on the previous one.

In [ ]:
# Setup: ensure project root is on sys.path
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np

pd.set_option('display.max_columns', 30)
pd.set_option('display.float_format', '{:.2f}'.format)
print('✅ Setup complete')

---
## Step 1: Data Ingestion & Harmonization

**Why**: Each database (DG, S&F, WG) has different column names and date formats.
We harmonize them into a single unified schema so downstream code is database-agnostic.

In [ ]:
from src.data.mock_generator import generate_all
from src.data.schema import DG_SCHEMA, SF_SCHEMA, WG_SCHEMA, harmonize_to_unified

# Generate synthetic data for all 3 databases
datasets = generate_all(save=True)

print(f"\nGenerated {len(datasets)} datasets:")
for name, df in datasets.items():
    print(f"  {name}: {len(df):,} rows × {len(df.columns)} cols")
    print(f"    Columns: {list(df.columns)}")

In [ ]:
# Harmonize each database to the unified schema
dg = harmonize_to_unified(datasets['DG'], DG_SCHEMA)
sf = harmonize_to_unified(datasets['SF'], SF_SCHEMA)
wg = harmonize_to_unified(datasets['WG'], WG_SCHEMA)

# Combine into a single DataFrame
df = pd.concat([dg, sf, wg], ignore_index=True)
df['date'] = pd.to_datetime(df['date'])

print(f"✅ Combined dataset: {len(df):,} rows")
print(f"Columns: {list(df.columns)}")
print(f"\nRetailers: {df['retailer'].unique()}")
print(f"Date range: {df['date'].min()} → {df['date'].max()}")
df.head()

---
## Step 2: Exploratory Data Analysis

**Why**: Understanding data structure informs every downstream choice — feature selection, model family, evaluation strategy.

In [ ]:
from src.utils.helpers import summarize_df

# Quick data profile
profile = summarize_df(df)
profile

In [ ]:
from src.visualization.plots import (
    plot_sales_trend, plot_seasonality_heatmap,
    plot_distribution, plot_rolling_average_comparison,
)

# Sales trend by retailer
fig = plot_sales_trend(df, target_col='qty_sold', color_col='retailer')
fig.show()

In [ ]:
# Seasonality heatmap
fig = plot_seasonality_heatmap(df, target_col='qty_sold')
fig.show()

In [ ]:
# Target distribution by retailer
fig = plot_distribution(df, target_col='qty_sold', group_col='retailer')
fig.show()

In [ ]:
# Rolling average comparison
fig = plot_rolling_average_comparison(df, target_col='qty_sold', window=8)
fig.show()

In [ ]:
# Missing data check
print('Missing data:')
for col in df.columns:
    null_pct = df[col].isna().mean() * 100
    if null_pct > 0:
        print(f'  {col}: {null_pct:.1f}% missing')

---
## Step 3: Data Validation & Distribution Assessment

**Why**: Choosing the right model for the right distribution is foundational.
A model optimized for normal residuals will perform poorly on heavily skewed data.
This step provides evidence-based model selection instead of guessing.

In [ ]:
from src.data.validation import generate_validation_report, print_validation_report

# Run comprehensive validation
report = generate_validation_report(
    df,
    target_col='qty_sold',
    date_col='date',
    group_cols=['retailer'],
)

# Print human-readable report with model recommendations
print_validation_report(report)

In [ ]:
# Per-retailer validation — different retailers may need different models
for retailer in df['retailer'].unique():
    r_df = df[df['retailer'] == retailer].copy()
    if r_df['qty_sold'].dropna().empty:
        print(f'{retailer}: no data — skipping')
        continue
    r_report = generate_validation_report(
        r_df, target_col='qty_sold', date_col='date', group_cols=['store']
    )
    top = r_report['recommendations'][0]
    print(f'{retailer}: {top.model_family} [{top.suitability}]')
    print(f'  {top.reasoning[:120]}...')
    print()

---
## Step 4: Feature Engineering

**Why**: Tree-based models cannot extrapolate — they need explicit lag and calendar
features to learn temporal patterns. All features use shifted values to prevent data leakage.

In [ ]:
from src.features.feature_engineering import build_features

# For speed, use a single retailer (DG) for the modeling steps
# In production, use all data for cross-learning
df_model = df[df['retailer'] == 'DG'].copy()
df_model = df_model.dropna(subset=['qty_sold'])

# Fill missing categorical columns for feature engineering
for col in ['product', 'category', 'department']:
    if col in df_model.columns:
        df_model[col] = df_model[col].fillna('unknown')

print(f'Working with DG data: {len(df_model):,} rows')

# Run full feature pipeline
df_features = build_features(
    df_model,
    target='qty_sold',
    lags=[1, 2, 4, 8, 12],
    rolling_windows=[4, 8],
    include_hierarchical=True,
)

new_cols = [c for c in df_features.columns if c not in df_model.columns]
print(f'✅ Added {len(new_cols)} features')
print(f'New columns: {new_cols}')

# Verify no leakage: first row lags should be NaN
print(f"\nLeakage check (should be NaN): lag_1 = {df_features['lag_1'].iloc[0]}")

---
## Step 5: Train/Test Split

**Why**: ❌ NEVER use random splits for time series. Future observations in the training set
cause data leakage → overly optimistic metrics that don’t reflect real performance.

In [ ]:
from src.config.config import TRAIN_CUTOFF_FRAC

# Sort by date
df_features = df_features.sort_values('date')

# Time-based split at 80th percentile
dates = df_features['date'].sort_values()
cutoff_date = dates.quantile(TRAIN_CUTOFF_FRAC)

train = df_features[df_features['date'] <= cutoff_date].copy()
test = df_features[df_features['date'] > cutoff_date].copy()

print(f'Train: {len(train):,} rows | {train["date"].min().date()} → {train["date"].max().date()}')
print(f'Test:  {len(test):,} rows  | {test["date"].min().date()} → {test["date"].max().date()}')
print(f'Cutoff: {cutoff_date.date()}')

In [ ]:
# Prepare feature matrix
exclude = {'date', 'qty_sold', 'net_sales', 'retailer', 'store', 'product',
           'category', 'department', 'city', 'state'}
feature_cols = [c for c in df_features.columns
                if c not in exclude and df_features[c].dtype in ['float64', 'int64', 'int32']]

X_train = train[feature_cols].fillna(0)
y_train = train['qty_sold'].fillna(0)
X_test = test[feature_cols].fillna(0)
y_test = test['qty_sold'].fillna(0)

print(f'Features: {len(feature_cols)}')
print(f'X_train: {X_train.shape}, X_test: {X_test.shape}')

---
## Step 6: Baseline Models

**Why**: Any ML model that can’t beat a naive baseline has negative value.
Baselines set the performance floor.

In [ ]:
from src.models.baseline_models import run_all_baselines
from src.evaluation.metrics import all_metrics

# Aggregate to a single series for baseline testing
train_agg = train.groupby('date')['qty_sold'].sum().sort_index()
test_agg = test.groupby('date')['qty_sold'].sum().sort_index()

# Run all baselines
horizon = len(test_agg)
forecasts = run_all_baselines(train_agg, horizon)

# Evaluate each baseline
baseline_results = []
for name, preds in forecasts.items():
    n = min(len(preds), len(test_agg))
    metrics = all_metrics(test_agg.values[:n], preds[:n])
    metrics['model'] = name
    baseline_results.append(metrics)

baseline_df = pd.DataFrame(baseline_results)
print('Baseline Results:')
baseline_df

---
## Step 7: Advanced Models

### XGBoost
**Why**: Handles non-linear relationships, missing values, and cross-learns across series.
Robust to skewed distributions and outliers (promotional spikes).

In [ ]:
try:
    from xgboost import XGBRegressor
    
    xgb_model = XGBRegressor(
        n_estimators=300, max_depth=6, learning_rate=0.05,
        subsample=0.8, colsample_bytree=0.8,
        random_state=42, n_jobs=-1,
    )
    xgb_model.fit(X_train, y_train, eval_set=[(X_test, y_test)], verbose=False)
    
    xgb_preds = np.maximum(0, xgb_model.predict(X_test))
    xgb_metrics = all_metrics(y_test.values, xgb_preds)
    xgb_metrics['model'] = 'XGBoost'
    print('✅ XGBoost trained')
    print(xgb_metrics)
except ImportError:
    print('⚠️  Install xgboost: pip install xgboost')
    xgb_metrics = None

### LightGBM
**Why**: Faster training than XGBoost, handles categorical features natively, often comparable accuracy.

In [ ]:
try:
    import lightgbm as lgb
    
    lgb_model = lgb.LGBMRegressor(
        n_estimators=300, max_depth=6, learning_rate=0.05,
        subsample=0.8, colsample_bytree=0.8,
        random_state=42, n_jobs=-1, verbosity=-1,
    )
    lgb_model.fit(X_train, y_train)
    
    lgb_preds = np.maximum(0, lgb_model.predict(X_test))
    lgb_metrics = all_metrics(y_test.values, lgb_preds)
    lgb_metrics['model'] = 'LightGBM'
    print('✅ LightGBM trained')
    print(lgb_metrics)
except ImportError:
    print('⚠️  Install lightgbm: pip install lightgbm')
    lgb_metrics = None

### ARIMA
**Why**: Best for interpretable, single-series analysis with statistical confidence intervals.
If the data is non-stationary (from Step 3), we apply differencing (d≥1).

In [ ]:
try:
    from statsmodels.tsa.arima.model import ARIMA
    
    # Use stationarity result from Step 3 to set differencing order
    d = 0 if report['stationarity']['is_stationary'] else 1
    
    arima_model = ARIMA(train_agg, order=(2, d, 2))
    arima_fitted = arima_model.fit()
    arima_forecast = arima_fitted.forecast(steps=len(test_agg))
    
    arima_metrics = all_metrics(test_agg.values, arima_forecast.values)
    arima_metrics['model'] = f'ARIMA(2,{d},2)'
    print(f'✅ ARIMA(2,{d},2) fitted')
    print(arima_metrics)
except ImportError:
    print('⚠️  Install statsmodels: pip install statsmodels')
    arima_metrics = None
except Exception as e:
    print(f'ARIMA error: {e}')
    arima_metrics = None

### Prophet
**Why**: Handles seasonality, trend changes, and holidays automatically. If distribution
is skewed (from Step 3), we apply log-transform before fitting.

In [ ]:
try:
    from prophet import Prophet
    
    prophet_df = train_agg.reset_index()
    prophet_df.columns = ['ds', 'y']
    
    # Apply log-transform if skewed
    skew = report['distribution']['skewness']
    if skew and skew > 1.0:
        prophet_df['y'] = np.log1p(prophet_df['y'])
        print(f'Applied log1p transform (skewness={skew:.2f})')
    
    m = Prophet(yearly_seasonality=True, weekly_seasonality=False, daily_seasonality=False)
    m.fit(prophet_df)
    
    future = m.make_future_dataframe(periods=len(test_agg), freq='D')
    forecast = m.predict(future)
    prophet_preds = forecast['yhat'].tail(len(test_agg)).values
    
    if skew and skew > 1.0:
        prophet_preds = np.expm1(prophet_preds)
    
    prophet_metrics = all_metrics(test_agg.values, prophet_preds)
    prophet_metrics['model'] = 'Prophet'
    print('✅ Prophet fitted')
    print(prophet_metrics)
except ImportError:
    print('⚠️  Prophet not installed (optional). Run: pip install prophet')
    prophet_metrics = None
except Exception as e:
    print(f'Prophet error: {e}')
    prophet_metrics = None

---
## Step 8: Hyperparameter Tuning (Optional)

**Why**: Default params are rarely optimal. Tuning can improve MAE by 10-30%.
Uncomment and run if Optuna is installed.

In [ ]:
# Uncomment to run hyperparameter tuning
# Requires: pip install optuna xgboost

# import optuna
# from xgboost import XGBRegressor
# from src.evaluation.metrics import mae as mae_fn
# 
# def objective(trial):
#     params = {
#         'max_depth': trial.suggest_int('max_depth', 3, 8),
#         'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
#         'n_estimators': trial.suggest_int('n_estimators', 100, 500),
#         'subsample': trial.suggest_float('subsample', 0.6, 1.0),
#         'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0),
#         'min_child_weight': trial.suggest_int('min_child_weight', 1, 20),
#         'random_state': 42, 'n_jobs': -1,
#     }
#     model = XGBRegressor(**params)
#     model.fit(X_train, y_train, verbose=False)
#     return mae_fn(y_test.values, model.predict(X_test))
# 
# study = optuna.create_study(direction='minimize')
# study.optimize(objective, n_trials=30)
# print(f'Best MAE: {study.best_value:.2f}')
# print(f'Best params: {study.best_params}')

print('ℹ️  Tuning is commented out by default — uncomment to run.')

---
## Step 9: Backtesting & Evaluation

**Why**: A single train/test split gives one data point about model quality.
Rolling-origin backtesting gives multiple realistic evaluations across different time periods.

In [ ]:
from src.evaluation.backtesting import BacktestConfig, rolling_origin_backtest, summarize_backtest

config = BacktestConfig(
    date_col='date',
    target_col='qty_sold',
    horizon=13,
    n_splits=3,
    min_train_periods=52,
)

# Aggregate series for backtesting
agg_df = df_features.groupby('date').agg({'qty_sold': 'sum'}).reset_index()

# Simple forecast function: use rolling mean
def simple_forecast(train_df, test_df):
    avg = train_df['qty_sold'].tail(13).mean()
    return np.full(len(test_df), avg)

try:
    results = rolling_origin_backtest(agg_df, simple_forecast, config)
    summary = summarize_backtest(results)
    print('✅ Backtest Results:')
    display(summary)
except ValueError as e:
    print(f'Backtest skipped: {e}')

---
## Step 10: Model Comparison & Final Selection

**Why**: Different models may excel at different segments. Compare all side-by-side
and select based on accuracy, robustness, segment fairness, and interpretability.

In [ ]:
from src.visualization.plots import plot_model_comparison

# Collect all model results
all_results = baseline_results.copy()

for m in [xgb_metrics, lgb_metrics, arima_metrics, prophet_metrics]:
    if m is not None:
        all_results.append(m)

comparison = pd.DataFrame(all_results)
comparison = comparison.sort_values('MAE')

print('🏆 Model Comparison (sorted by MAE):')
display(comparison[['model', 'MAE', 'RMSE', 'MAPE', 'sMAPE', 'WAPE', 'Bias']].reset_index(drop=True))

In [ ]:
# Visual comparison
if len(comparison) > 0:
    fig = plot_model_comparison(comparison, metric='MAE')
    fig.show()

In [ ]:
# Feature importance (if XGBoost was trained)
try:
    from src.models.ml_models import feature_importance_df
    from src.visualization.plots import plot_feature_importance
    
    fi = feature_importance_df(xgb_model, feature_cols)
    fig = plot_feature_importance(fi, top_n=15)
    fig.show()
    
    # Lag feature dominance check
    lag_imp = fi[fi['feature'].str.startswith('lag_')]['importance'].sum()
    total = fi['importance'].sum()
    print(f'\nLag features: {lag_imp/total:.1%} of total importance')
except:
    print('Feature importance requires XGBoost model')

---
## ✅ Done!

You've completed the full ML pipeline. Key takeaways:

1. **Always validate your data** before choosing a model (Step 3)
2. **Never use random splits** for time series (Step 5)
3. **Always beat the baseline** before deploying any model (Step 6)
4. **Match the model to the data** — not the other way around (Step 7)
5. **Use backtesting** for realistic evaluation (Step 9)

For the full reasoning docs, see [docs/ml_pipeline_guide.md](../docs/ml_pipeline_guide.md).